In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from getdist import plots, MCSamples
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'text.usetex': False
})
chain_file = 'chains/chain_withprior.txt'
with open(chain_file, 'r') as f:
    header = f.readline().replace('#', '').strip().split()

df = pd.read_csv(chain_file, sep='\s+', comment='#', names=header)
df = df.dropna()
log_w = df['log_weight'].values
weights = np.exp(log_w - np.max(log_w))

In [2]:
z1, z2 = 0.38, 0.61
omega_m = df['COSMOLOGICAL_PARAMETERS--OMEGA_M'].values
jhat1 = df['J_bin_bias--jhat1'].values
jhat2 = df['J_bin_bias--jhat2'].values
fsigma81 = df['LSS_PARAMETERS--FSIGMA_8_BIN_1'].values
fsigma82 = df['LSS_PARAMETERS--FSIGMA_8_BIN_2'].values
E_G_1 = (jhat1 / fsigma81) * (omega_m + (1.0 - omega_m) / (1.0 + z1)**3)
E_G_2 = (jhat2 / fsigma82) * (omega_m + (1.0 - omega_m) / (1.0 + z2)**3)
df['DERIVED--E_G_1'] = E_G_1
df['DERIVED--E_G_2'] = E_G_2

params_to_plot = [
    'COSMOLOGICAL_PARAMETERS--OMEGA_M',
    'COSMOLOGICAL_PARAMETERS--S_8',
    'COSMOLOGICAL_PARAMETERS--SIGMA_8',
    'J_bin_bias--jhat1',
    'J_bin_bias--jhat2',
    'LSS_PARAMETERS--FSIGMA_8_BIN_1',
    'LSS_PARAMETERS--FSIGMA_8_BIN_2',
    'DERIVED--E_G_1',
    'DERIVED--E_G_2'
]
labels = [
    r'\Omega_m',
    r'S_8',
    r'\sigma_8',
    r'\hat{J}_1',
    r'\hat{J}_2',
    r'f\sigma_{8,1}',
    r'f\sigma_{8,2}',
    r'E_{G,1}',
    r'E_{G,2}'
]
samples = np.vstack([df[p].values for p in params_to_plot]).T
param_names = [p.replace('-', '_') for p in params_to_plot]
mc_samples = MCSamples(samples=samples, weights=weights, names=param_names, labels=labels, label='KiDS+BOSS 2x2pt')

stats = mc_samples.getMargeStats() # marginalize over all parameters
target_vars = [
    ('J_bin_bias__jhat1',           'J_hat (z=0.38)'),
    ('LSS_PARAMETERS__FSIGMA_8_BIN_1', 'f.sigma8 (z=0.38)'),
    ('DERIVED__E_G_1',              'E_G (z=0.38)'),
    ('J_bin_bias__jhat2',           'J_hat (z=0.61)'),
    ('LSS_PARAMETERS__FSIGMA_8_BIN_2', 'f.sigma8 (z=0.61)'),
    ('DERIVED__E_G_2',              'E_G (z=0.61)')
]
table_data = []
for p_name, display_name in target_vars:
    par_stat = stats.parWithName(p_name)
    mean = par_stat.mean
    err_sym = par_stat.err
    table_data.append({
        'Parameter': display_name,
        'Mean': f"{mean:.3f}",
        '1-sigma Error': f"± {err_sym:.3f}",
        'Lower 68%': f"{par_stat.limits[0].lower:.3f}",
        'Upper 68%': f"{par_stat.limits[0].upper:.3f}"
    })
results_df = pd.DataFrame(table_data)
print(results_df.to_string(index=False))

g = plots.get_subplot_plotter()
g.settings.axes_fontsize = 14
g.settings.lab_fontsize = 16
g.settings.legend_fontsize = 20
params_for_corner = [
    'COSMOLOGICAL_PARAMETERS__OMEGA_M',
    'COSMOLOGICAL_PARAMETERS__S_8',
    'J_bin_bias__jhat1',
    'J_bin_bias__jhat2',
    'DERIVED__E_G_1',
    'DERIVED__E_G_2'
]
g.triangle_plot([mc_samples], params=params_for_corner, filled=True, contour_colors=['#006eb8'], title_limit=1)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
plt.savefig('fig/cosmology_constraints_2x2pt_withprior.pdf')
plt.show()

Removed no burn in
        Parameter  Mean 1-sigma Error Lower 68% Upper 68%
   J_hat (z=0.38) 0.360       ± 0.038     0.320     0.395
f.sigma8 (z=0.38) 0.478       ± 0.007     0.471     0.485
     E_G (z=0.38) 0.431       ± 0.046     0.383     0.474
   J_hat (z=0.61) 0.325       ± 0.042     0.283     0.367
f.sigma8 (z=0.61) 0.472       ± 0.007     0.465     0.478
     E_G (z=0.61) 0.328       ± 0.043     0.286     0.371


In [3]:
from matplotlib.ticker import MultipleLocator

def growth_factor(z, Om0):
    Oml0 = 1.0 - Om0
    a = 1.0 / (1.0 + z)
    E2 = Om0 * (1 + z)**3 + Oml0
    Om_z = Om0 * (1 + z)**3 / E2
    Oml_z = Oml0 / E2
    def g(om, oml):
        return (5.0 / 2.0) * om / (om**(4.0/7.0) - oml + (1.0 + om/2.0)*(1.0 + oml/70.0))
    D_z = (g(Om_z, Oml_z) / g(Om0, Oml0)) * a
    return D_z, Om_z

def J_hat_theoretical(z_arr, Om0, sigma8_0):
    J_hat = np.zeros_like(z_arr)
    for i, z in enumerate(z_arr):
        D_z, Om_z = growth_factor(z, Om0)
        J_hat[i] = Om_z * sigma8_0 * D_z
    return J_hat

z_theory = np.linspace(0.15, 0.85, 100)

Om0_planck, sigma8_planck = 0.315, 0.811
Om0_planck_err, sigma8_planck_err = 0.007, 0.006
J_theory_planck = J_hat_theoretical(z_theory, Om0_planck, sigma8_planck)
err_planck = J_theory_planck * np.sqrt((Om0_planck_err/Om0_planck)**2 + (sigma8_planck_err/sigma8_planck)**2)

par_omega_m = stats.parWithName('COSMOLOGICAL_PARAMETERS__OMEGA_M')
par_sigma8 = stats.parWithName('COSMOLOGICAL_PARAMETERS__SIGMA_8')
Om0_kids = par_omega_m.mean
sigma8_kids = par_sigma8.mean
Om0_kids_err = (par_omega_m.limits[0].upper - par_omega_m.limits[0].lower) / 2.0
sigma8_kids_err = (par_sigma8.limits[0].upper - par_sigma8.limits[0].lower) / 2.0
J_theory_kids = J_hat_theoretical(z_theory, Om0_kids, sigma8_kids)
err_kids_curve = J_theory_kids * np.sqrt((Om0_kids_err/Om0_kids)**2 + (sigma8_kids_err/sigma8_kids)**2)

z_kids = np.array([z1, z2])
par_j1 = stats.parWithName('J_bin_bias__jhat1')
par_j2 = stats.parWithName('J_bin_bias__jhat2')
J_kids = np.array([par_j1.mean, par_j2.mean])
err_kids = np.array([par_j1.err, par_j2.err])

z_des = np.array([0.295, 0.467, 0.626, 0.771])
J_des = np.array([0.325, 0.333, 0.387, 0.354])
err_des = np.array([0.015, 0.018, 0.027, 0.035])

J_pred_lcdm = np.array([
    J_hat_theoretical(np.array([z1]), Om0_kids, sigma8_kids)[0],
    J_hat_theoretical(np.array([z2]), Om0_kids, sigma8_kids)[0],
])
tension_J = np.abs(J_kids - J_pred_lcdm) / err_kids

print("=" * 60)
print("J_hat(z) Measurements vs LCDM Predictions (Om_m, sigma_8 from chain)")
print("=" * 60)
for i, z in enumerate(z_kids):
    print(f"Bin {i + 1} (z = {z:.2f}):")
    print(f"  Measured J_hat : {J_kids[i]:.3f} +/- {err_kids[i]:.3f}")
    print(f"  Predicted LCDM : {J_pred_lcdm[i]:.3f}")
    print(f"  Tension        : {tension_J[i]:.2f} sigma")
print("=" * 60)

fig, ax = plt.subplots(figsize=(8, 6))
# ax.plot(z_theory, J_theory_planck, color='black', linestyle='--', linewidth=2, label=r'$\Lambda$CDM (Planck 18)')
# ax.fill_between(z_theory, J_theory_planck - err_planck, J_theory_planck + err_planck, color='black', alpha=0.1, edgecolor='none')
ax.plot(z_theory, J_theory_kids, color='royalblue', linestyle='-.', linewidth=2, label=r'$\Lambda$CDM, with prior')
ax.fill_between(z_theory, J_theory_kids - err_kids_curve, J_theory_kids + err_kids_curve, color='royalblue', alpha=0.15, edgecolor='none')
ax.errorbar(z_des, J_des, yerr=err_des, fmt='o', color='gray', alpha=0.7, markersize=8, capsize=4, elinewidth=1.5, markeredgewidth=1.5, label='DES MagLim (Tutusaus+24)')
ax.errorbar(z_kids, J_kids, yerr=err_kids, fmt='*', color='crimson', markersize=15, capsize=4, elinewidth=2, markeredgewidth=2, zorder=5, label='KiDS-1000+BOSS+2dFlenS')
ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'Weyl Potential Evolution $\hat{J}(z)$')
ax.set_xlim(0.2, 0.85)
ax.set_ylim(0.24, 0.44)
ax.tick_params(axis='both', which='major', length=6, direction='in')
ax.tick_params(axis='both', which='minor', length=3, direction='in')
ax.xaxis.set_minor_locator(MultipleLocator(0.05))
ax.yaxis.set_minor_locator(MultipleLocator(0.01))
ax.legend(loc='best', frameon=True, edgecolor='black', fancybox=False)
ax.axvspan(0.57, 0.7, color='orange', alpha=0.1)
# ax.text(0.615, 0.25, r'"Lensing is Low" Region', color='darkorange', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('fig/J_hat_evolution_comparison_withprior.pdf')
plt.show()

J_hat(z) Measurements vs LCDM Predictions (Om_m, sigma_8 from chain)
Bin 1 (z = 0.38):
  Measured J_hat : 0.360 +/- 0.038
  Predicted LCDM : 0.363
  Tension        : 0.10 sigma
Bin 2 (z = 0.61):
  Measured J_hat : 0.325 +/- 0.042
  Predicted LCDM : 0.389
  Tension        : 1.54 sigma


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

def E_G_theoretical(z_arr, Om0):
    Oml0 = 1.0 - Om0
    E_G = np.zeros_like(z_arr)
    for i, z in enumerate(z_arr):
        E2 = Om0 * (1 + z)**3 + Oml0
        Om_z = Om0 * (1 + z)**3 / E2
        f_z = Om_z**0.55
        E_G[i] = Om0 / f_z
    return E_G

z_theory = np.linspace(0.15, 0.85, 100)
Om0_planck = 0.315
EG_theory_planck = E_G_theoretical(z_theory, Om0_planck)

z_kids = np.array([0.38, 0.61])
EG_kids = np.array([
    np.average(df['DERIVED--E_G_1'], weights=weights),
    np.average(df['DERIVED--E_G_2'], weights=weights)
])
err_kids = np.sqrt([
    np.cov(df['DERIVED--E_G_1'], aweights=weights),
    np.cov(df['DERIVED--E_G_2'], aweights=weights)
])

Om0_kids = np.average(df['COSMOLOGICAL_PARAMETERS--OMEGA_M'], weights=weights)
EG_theory_kids = E_G_theoretical(z_theory, Om0_kids)

EG_pred_exact = E_G_theoretical(z_kids, Om0_kids)
tension = np.abs(EG_kids - EG_pred_exact) / err_kids

print("="*60)
print("E_G(z) Measurements vs GR Predictions")
print("="*60)
for i, z in enumerate(z_kids):
    print(f"Bin {i+1} (z = {z:.2f}):")
    print(f"  Measured E_G : {EG_kids[i]:.3f} +/- {err_kids[i]:.3f}")
    print(f"  Predicted GR : {EG_pred_exact[i]:.3f}")
    print(f"  Tension      : {tension[i]:.2f} sigma")
print("="*60)

z_des = np.array([0.295, 0.467, 0.626, 0.771])
EG_des = np.array([0.447, 0.378, 0.396, 0.345])
err_des = np.array([0.027, 0.026, 0.033, 0.039])

z_lit = np.array([0.37, 0.32, 0.57, 0.27, 0.31, 0.55, 0.57, 0.57, 0.316, 0.555])
EG_lit = np.array([0.392, 0.48, 0.30, 0.43, 0.27, 0.26, 0.420, 0.24, 0.40, 0.36])

err_lit_down = np.array([0.065, 0.10, 0.07, 0.13, 0.08, 0.07, 0.056, 0.06, 0.09, 0.05])
err_lit_up   = np.array([0.065, 0.10, 0.07, 0.13, 0.08, 0.07, 0.056, 0.06, 0.11, 0.06])
err_lit =[err_lit_down, err_lit_up]

fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(z_lit, EG_lit, yerr=err_lit, fmt='s', color='silver', alpha=0.8, markersize=6, capsize=3, elinewidth=1.2, markeredgewidth=1.2, markerfacecolor='white', zorder=1, label='From reference')
ax.errorbar(z_des, EG_des, yerr=err_des, fmt='D', color='steelblue', alpha=0.85, markersize=7, capsize=4, elinewidth=1.5, markeredgewidth=1.5, zorder=2, label='DES MagLim (Grimm+24)')
ax.plot(z_theory, EG_theory_kids, color='black', linestyle='--', linewidth=2, zorder=3, label=r'GR prediction (Planck2018)')
ax.errorbar(z_kids, EG_kids, yerr=err_kids, fmt='*', color='crimson', markersize=16, capsize=5, elinewidth=2.5, markeredgewidth=2, zorder=5, label='KiDS-1000+BOSS+2dFlenS')

ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'Gravitational Slip Statistic $E_G(z)$')
ax.set_xlim(0.18, 0.82)
ax.set_ylim(0.15, 0.65)
ax.tick_params(axis='both', which='major', length=6, direction='in', top=True, right=True)
ax.tick_params(axis='both', which='minor', length=3, direction='in', top=True, right=True)
ax.xaxis.set_minor_locator(MultipleLocator(0.05))
ax.yaxis.set_minor_locator(MultipleLocator(0.025))
ax.legend(loc='best', frameon=True, edgecolor='black', fancybox=False)
plt.tight_layout()
plt.savefig('fig/EG_compilation_withprior.pdf')
plt.show()

E_G(z) Measurements vs GR Predictions
Bin 1 (z = 0.38):
  Measured E_G : 0.431 +/- 0.046
  Predicted GR : 0.436
  Tension      : 0.10 sigma
Bin 2 (z = 0.61):
  Measured E_G : 0.328 +/- 0.043
  Predicted GR : 0.394
  Tension      : 1.53 sigma


In [5]:
j_samples = df[['J_bin_bias--jhat1', 'J_bin_bias--jhat2']].values.T

j_mean = np.average(j_samples, axis=1, weights=weights)
j_cov = np.cov(j_samples, aweights=weights)

print(j_mean)
print(j_cov)

[0.35951033 0.32478106]
[[0.00146212 0.00018427]
 [0.00018427 0.0017448 ]]


In [6]:
from scipy.integrate import quad
from scipy.stats import chi2

z_obs = np.array([0.38, 0.61])
j_obs = j_mean
err_obs = np.array([0.035, 0.037])
inv_cov = np.linalg.inv(j_cov)
Om0 = Om0_kids
sigma8_0 = sigma8_kids
Oml0 = 1.0 - Om0

def E_z(z): return np.sqrt(Om0 * (1 + z) ** 3 + Oml0)
def Om_z(z): return Om0 * (1 + z) ** 3 / E_z(z) ** 2
def Oml_z(z): return Oml0 / E_z(z) ** 2
def growth_integrand(z): return (1 + z) / E_z(z) ** 3
def D_z_norm(z): return (E_z(z) * quad(growth_integrand, z, np.inf)[0]) / quad(growth_integrand, 0, np.inf)[0]
def J_LCDM(z): return Om_z(z) * (sigma8_0 * D_z_norm(z))
def g_standard(z): return float(Oml_z(z))
def g_constant(z): return 1.0 if (0 <= z <= 1) else 0.0
def g_exponential(z): return float(np.exp(1 + z)) if (0 <= z <= 1) else 0.0

models = {'Standard': g_standard, 'Constant': g_constant, 'Exponential': g_exponential}

def fit_Sigma0_analytical(g_func):
    J0 = np.array([J_LCDM(z) for z in z_obs])
    G = np.array([g_func(z) for z in z_obs])
    X = G * J0
    Delta = j_obs - J0
    Fisher = X.T @ inv_cov @ X
    best_Sigma_0 = (X.T @ inv_cov @ Delta) / Fisher
    sigma_err = 1.0 / np.sqrt(Fisher)
    res_vec = j_obs - (J0 + best_Sigma_0 * X)
    min_chi2 = res_vec.T @ inv_cov @ res_vec
    return float(best_Sigma_0), float(sigma_err), float(min_chi2)

dof_LCDM = len(z_obs)
j_th_LCDM = np.array([J_LCDM(z) for z in z_obs])
delta_LCDM = j_obs - j_th_LCDM
chi2_LCDM = delta_LCDM.T @ inv_cov @ delta_LCDM
print(f"[LambdaCDM  ] Chi2: {chi2_LCDM:.2f} | Reduced Chi2: {chi2_LCDM/dof_LCDM:.2f} | p-value: {chi2.sf(chi2_LCDM, dof_LCDM):.3f}")

dof_mod = len(z_obs) - 1
results = {}
for name, func in models.items():
    best_val, err, min_c2 = fit_Sigma0_analytical(func)
    results[name] = {'best': best_val, 'err': err}
    print(f"[{name:<11}] Sigma_0 = {best_val:>6.3f} ± {err:.3f} | Tension: {abs(best_val)/err:.1f} sigma | Reduced Chi2: {min_c2/dof_mod:.2f} | p-value: {chi2.sf(min_c2, dof_mod):.3f}")

plt.rcParams['font.family'] = 'serif'
fig, ax = plt.subplots(figsize=(8, 6))
z_plot = np.linspace(0.15, 0.85, 200)
j_base = np.array([J_LCDM(z) for z in z_plot])
ax.plot(z_plot, j_base, color='black', linewidth=2.5, zorder=5, label=r'GR ($\Lambda$CDM, $\Sigma_0=0$)')
# Softer blue-cyan-purple palette
colors = ['crimson', 'forestgreen', 'darkorange']
linestyles = ['--', '-.', ':']
labels =[r'Standard: $g(z)=\Omega_\Lambda(z)$', r'Constant: $g(z)=1$', r'Exponential: $g(z)=e^{1+z}$']

for i, (name, func) in enumerate(models.items()):
    best_S0, err_S0 = results[name]['best'], results[name]['err']
    j_mod_best = np.array([(1.0 + best_S0 * func(z)) * J_LCDM(z) for z in z_plot])
    j_mod_up = np.array([(1.0 + (best_S0 + err_S0) * func(z)) * J_LCDM(z) for z in z_plot])
    j_mod_down = np.array([(1.0 + (best_S0 - err_S0) * func(z)) * J_LCDM(z) for z in z_plot])
    ax.plot(z_plot, j_mod_best, color=colors[i], linestyle=linestyles[i], linewidth=2, zorder=4, label=labels[i])
    ax.fill_between(z_plot, j_mod_down, j_mod_up, color=colors[i], alpha=0.18, edgecolor='none', zorder=2)

ax.errorbar(z_obs, j_obs, yerr=err_obs, fmt='s', color='#2F3E63', markersize=8, capsize=5, elinewidth=2.5, markeredgewidth=2, zorder=6, label='KiDS-1000+BOSS+2dFlenS')
ax.set_xlabel(r'Redshift $z$', fontsize=16)
ax.set_ylabel(r'Weyl Potential Evolution $\hat{J}(z)$', fontsize=16)
ax.set_xlim(0.18, 0.8)
ax.set_ylim(0.23, 0.4)
ax.tick_params(axis='both', which='major', length=6, direction='in', top=True, right=True)
ax.tick_params(axis='both', which='minor', length=3, direction='in', top=True, right=True)
ax.xaxis.set_minor_locator(MultipleLocator(0.05))
ax.yaxis.set_minor_locator(MultipleLocator(0.01))
ax.legend(loc='lower left', frameon=True, edgecolor='black', fancybox=False)
plt.tight_layout()
plt.savefig('fig/J_hat_evolution_analytical_errors_withprior.pdf')
plt.show()

[LambdaCDM  ] Chi2: 2.34 | Reduced Chi2: 1.17 | p-value: 0.311
[Standard   ] Sigma_0 = -0.173 ± 0.195 | Tension: 0.9 sigma | Reduced Chi2: 1.55 | p-value: 0.213
[Constant   ] Sigma_0 = -0.085 ± 0.080 | Tension: 1.1 sigma | Reduced Chi2: 1.20 | p-value: 0.273
[Exponential] Sigma_0 = -0.021 ± 0.018 | Tension: 1.2 sigma | Reduced Chi2: 0.91 | p-value: 0.341


In [7]:
om = df['COSMOLOGICAL_PARAMETERS--OMEGA_M'].values
s8_c = df['COSMOLOGICAL_PARAMETERS--SIGMA_8'].values
j1 = df['J_bin_bias--jhat1'].values
j2 = df['J_bin_bias--jhat2'].values

def get_x(z, o):
    l = 1.0 - o
    e = o * (1 + z)**3 + l
    oz = o * (1 + z)**3 / e
    lz = l / e
    g = lambda m, ml: 2.5 * m / (m**(4/7) - ml + (1 + m/2) * (1 + ml/70))
    return oz * (g(oz, lz) / g(o, l)) / (1 + z)

x1 = get_x(0.38, om)
x2 = get_x(0.61, om)

s8_f1 = j1 / x1
s8_f2 = j2 / x2

mc = MCSamples(samples=np.column_stack([s8_c, s8_f1, s8_f2]), weights=weights, names=['s8c', 's8f1', 's8f2'])
st = mc.getMargeStats()

def get_res(n):
    s = st.parWithName(n)
    return s.mean, s.mean - s.limits[0].lower, s.limits[0].upper - s.mean

m_c, d_c, u_c = get_res('s8c')
m_1, d_1, u_1 = get_res('s8f1')
m_2, d_2, u_2 = get_res('s8f2')

t1 = abs(m_c - m_1) / np.sqrt((u_1 if m_1 < m_c else d_1)**2 + (d_c if m_c > m_1 else u_c)**2)
t2 = abs(m_c - m_2) / np.sqrt((u_2 if m_2 < m_c else d_2)**2 + (d_c if m_c > m_2 else u_c)**2)

print("="*75)
print(" Independent Redshift Test for Sigma_8 (Lensing vs Clustering)")
print("="*75)
print(f"[Global (BOSS RSD)  ] sigma8_cosmo = {m_c:.3f} +{u_c:.3f} -{d_c:.3f}")
print(f"[Bin 1 (z=0.38) Lens] sigma8_fit_1 = {m_1:.3f} +{u_1:.3f} -{d_1:.3f} | Tension = {t1:.2f} sigma")
print(f"[Bin 2 (z=0.61) Lens] sigma8_fit_2 = {m_2:.3f} +{u_2:.3f} -{d_2:.3f} | Tension = {t2:.2f} sigma")
print("="*75)

Removed no burn in
 Independent Redshift Test for Sigma_8 (Lensing vs Clustering)
[Global (BOSS RSD)  ] sigma8_cosmo = 0.815 +0.012 -0.012
[Bin 1 (z=0.38) Lens] sigma8_fit_1 = 0.807 +0.080 -0.089 | Tension = 0.10 sigma
[Bin 2 (z=0.61) Lens] sigma8_fit_2 = 0.680 +0.088 -0.088 | Tension = 1.52 sigma
